**Метрики ранжирования: Uplift@K, Lift@K, Qini, AUUC, Cumulative Gain**

Представьте: вы запустили email-рассылку со скидкой 30%, push-уведомление или акцию в приложении.  
Обычная модель классификации скажет: «разошлите тем, у кого вероятность покупки > 0.4».  
Но многие из них и без скидки бы купили (это Sure Things).  
А кто-то вообще никогда не купит, даже со скидкой 90% (Lost Causes).

Uplift-модель должна отвечать на вопрос:  
**«Кому именно акция добавила покупки, которых бы не было без неё?»**

Но как понять, хорошая ли у нас uplift-модель?  

Мы разберём  популярные метрики, которые **используются в маркетинге**:

1. Uplift@K / Lift@K  
2. Cumulative Gain (Incremental conversions)  
3. Qini curve → Qini coefficient / AUQC  
4. AUUC (Area Under Uplift Curve)  
5. Cumulative Incremental Response Rate (CIRR)

Покажем на синтетическом датасете, как выглядят графики и что они значат на практике.

**Краткая теория — что мы вообще хотим измерить**

В uplift-задаче у нас есть:

- treatment (T = 1 — акция, T = 0 — контроль)  
- outcome (Y = 1 — покупка/конверсия, Y = 0 — нет)  
- predicted uplift τ̂ = вероятность покупки с акцией − вероятность покупки без акции

Мы **сортируем** всех клиентов по убыванию predicted uplift и смотрим:

- сколько **дополнительных** покупок мы получили, разослав акцию топ-10%, топ-20% и т.д.  
- насколько мы лучше **случайной** рассылки  
- насколько мы близки к **идеальному** таргетингу (знали бы истинный uplift)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from typing import List, Dict, Tuple, Set
import warnings
warnings.filterwarnings('ignore')

# Совместимость с NumPy 2.x: np.trapz удалён, используем np.trapezoid.
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid


In [ ]:
np.random.seed(42)

n_samples = 20000

age = np.random.normal(40, 10, n_samples).clip(18, 75)
income = np.random.normal(60000, 20000, n_samples).clip(10000, 150000)
recency = np.random.exponential(30, n_samples).clip(0, 365)  # дней с последнего визита

treatment = np.random.binomial(1, 0.5, n_samples)

base_prob = 1 / (1 + np.exp(-(-3 + 0.04*age - 0.000008*income + 0.012*recency)))
persuadable_boost = np.where((age < 38) & (income > 55000), 0.22, 0.035)

prob = base_prob + persuadable_boost * treatment
prob = np.clip(prob, 0, 1)

conversion = np.random.binomial(1, prob)

df = pd.DataFrame({
    'age': age.astype(int),
    'income': income.astype(int),
    'recency': recency.astype(int),
    'treatment': treatment,
    'conversion': conversion
})

print("Размер датасета:", df.shape)
print("Общая конверсия:", df['conversion'].mean())
print("\nКонверсия по группам:")
print(df.groupby('treatment')['conversion'].mean())

Простая uplift-модель (двухмодельный подход — baseline)

Обучим две логистические регрессии: одну на treatment=1, другую на treatment=0.

In [ ]:
train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df['treatment'])

X_cols = ['age', 'income', 'recency']

model_t = LogisticRegression(max_iter=1000).fit(
    train[train['treatment']==1][X_cols],
    train[train['treatment']==1]['conversion']
)

model_c = LogisticRegression(max_iter=1000).fit(
    train[train['treatment']==0][X_cols],
    train[train['treatment']==0]['conversion']
)

test['pred_t'] = model_t.predict_proba(test[X_cols])[:, 1]
test['pred_c'] = model_c.predict_proba(test[X_cols])[:, 1]
test['uplift'] = test['pred_t'] - test['pred_c']

# случайный uplift для сравнения
test['random_uplift'] = np.random.uniform(0, 0.4, len(test))

**Метрика 1: Uplift@K — самый понятный бизнес-показатель**

Сортируем по predicted uplift → берём топ-K% → считаем разницу конверсий (treatment − control)

In [ ]:
def uplift_at_k(df, uplift_col='uplift', k=0.1):
    df_sorted = df.sort_values(uplift_col, ascending=False)
    n_k = int(len(df_sorted) * k)
    top_k = df_sorted.iloc[:n_k]
    conv_t = top_k[top_k['treatment']==1]['conversion'].mean()
    conv_c = top_k[top_k['treatment']==0]['conversion'].mean()
    return conv_t - conv_c if conv_t and conv_c else 0

print(f"Uplift@10% (модель)  = {uplift_at_k(test, 'uplift', 0.1):.4f}")
print(f"Uplift@30% (модель)  = {uplift_at_k(test, 'uplift', 0.3):.4f}")
print(f"Uplift@10% (random)  = {uplift_at_k(test, 'random_uplift', 0.1):.4f}")

**Метрика 2: Cumulative Gain Curve (Qini-подобная)**

Накопленный дополнительный эффект по мере таргетинга всё большего % клиентов

In [ ]:
def cumulative_gain_curve(df, uplift_col='uplift'):
    df = df.copy().sort_values(uplift_col, ascending=False).reset_index(drop=True)
    df['cum_treated'] = df['treatment'].cumsum()
    df['cum_control'] = (1 - df['treatment']).cumsum()

    df['conv_t'] = df['conversion'] * df['treatment']
    df['conv_c'] = df['conversion'] * (1 - df['treatment'])

    df['cum_conv_t'] = df['conv_t'].cumsum()
    df['cum_conv_c'] = df['conv_c'].cumsum()

    df['gain'] = (df['cum_conv_t'] / df['cum_treated'].replace(0, np.nan) -
                  df['cum_conv_c'] / df['cum_control'].replace(0, np.nan)) * (df.index + 1) / len(df)

    return df['gain'].fillna(0).values

percent = np.linspace(0, 1, len(test))

gain_model = cumulative_gain_curve(test, 'uplift')
gain_random = cumulative_gain_curve(test, 'random_uplift')

plt.figure(figsize=(10, 6))
plt.plot(percent * 100, gain_model, label='Модель')
plt.plot(percent * 100, gain_random, '--', label='Случайно')
plt.xlabel('Процент таргетируемых клиентов (%)')
plt.ylabel('Cumulative Gain (доп. конверсий)')
plt.title('Cumulative Gain Curve')
plt.legend()
plt.grid(True)
plt.show()

**Метрика 3: Qini coefficient / AUQC**

Площадь между кривой модели и кривой рандома (чем больше — тем лучше)

In [ ]:
# Простая аппроксимация AUQC (можно использовать scikit-uplift для точного расчёта)
def approx_auqc(gain):
    return np.trapz(gain, dx=1/len(gain))

auqc_model = approx_auqc(gain_model)
auqc_random = approx_auqc(gain_random)

print(f"Приближённый AUQC (модель) = {auqc_model:.4f}")
print(f"Приближённый AUQC (random)  = {auqc_random:.4f}")

## **Ключевые выводы**

- Обычные метрики (accuracy, AUC) **не показывают** ценность uplift-модели  
- **Uplift@K** — самый понятный бизнесу показатель: «сколько допродаж даст топ-10%»  
- **Cumulative Gain** показывает, где кампания насыщается  
- **Qini / AUQC** — как ROC-AUC, но для uplift: насколько мы лучше рандома  
- Хорошая модель сильно выгибает кривую вверх в начале и даёт AUQC > 0.05–0.15 (зависит от ATE)

## Задания

1. Увеличить размер датасета до 100 000 строк и посмотреть, как изменятся Uplift@K и кривая.  
2. Замените логистическую регрессию на CatBoost (или XGBoost) → улучшатся ли метрики?  
3. Добавить шум в predicted uplift (например, умножить на 0.5 + случайный шум) — как сильно упадёт Qini?  
4. Посчитайте Uplift@K для k = 0.05, 0.1, 0.2, 0.5. Где оптимально остановить кампанию?  
5. Постройте Qini-кривую для **истинного** uplift (если бы мы знали настоящий persuadable_boost) — это будет «потолок».

In [ ]:
# Решение заданий по uplift/Qini
from xgboost import XGBClassifier


def make_uplift_dataset(n_samples=100000, random_state=42):
    rng = np.random.default_rng(random_state)
    age = rng.normal(40, 10, n_samples).clip(18, 75)
    income = rng.normal(60000, 20000, n_samples).clip(10000, 150000)
    recency = rng.exponential(30, n_samples).clip(0, 365)
    treatment = rng.binomial(1, 0.5, n_samples)

    base_prob = 1 / (1 + np.exp(-(-3 + 0.04 * age - 0.000008 * income + 0.012 * recency)))
    true_uplift = np.where((age < 38) & (income > 55000), 0.22, 0.035)
    prob = np.clip(base_prob + true_uplift * treatment, 0, 1)
    conversion = rng.binomial(1, prob)

    return pd.DataFrame({
        "age": age.astype(int),
        "income": income.astype(int),
        "recency": recency.astype(int),
        "treatment": treatment,
        "conversion": conversion,
        "true_uplift": true_uplift,
    })


def fit_two_model_uplift(data, model_factory, random_state=42):
    train_part, test_part = train_test_split(
        data,
        test_size=0.3,
        random_state=random_state,
        stratify=data["treatment"],
    )
    test_part = test_part.copy()

    model_t = model_factory()
    model_c = model_factory()

    model_t.fit(
        train_part[train_part["treatment"] == 1][X_cols],
        train_part[train_part["treatment"] == 1]["conversion"],
    )
    model_c.fit(
        train_part[train_part["treatment"] == 0][X_cols],
        train_part[train_part["treatment"] == 0]["conversion"],
    )

    test_part["pred_t"] = model_t.predict_proba(test_part[X_cols])[:, 1]
    test_part["pred_c"] = model_c.predict_proba(test_part[X_cols])[:, 1]
    test_part["uplift"] = test_part["pred_t"] - test_part["pred_c"]
    return test_part


def logistic_factory():
    return LogisticRegression(max_iter=1000)


def xgb_factory():
    return XGBClassifier(
        n_estimators=120,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=2,
        verbosity=0,
    )


def gain_at_k(gain_values, k):
    idx = max(0, min(len(gain_values) - 1, int(len(gain_values) * k) - 1))
    return gain_values[idx]


df_100k = make_uplift_dataset(100000, random_state=142)
test_100k_logit = fit_two_model_uplift(df_100k, logistic_factory, random_state=42)
gain_100k_logit = cumulative_gain_curve(test_100k_logit, "uplift")

comparison_100k = pd.DataFrame({
    "dataset/model": ["20k Logistic", "100k Logistic"],
    "Uplift@10%": [
        uplift_at_k(test, "uplift", 0.10),
        uplift_at_k(test_100k_logit, "uplift", 0.10),
    ],
    "Uplift@30%": [
        uplift_at_k(test, "uplift", 0.30),
        uplift_at_k(test_100k_logit, "uplift", 0.30),
    ],
    "AUQC": [approx_auqc(gain_model), approx_auqc(gain_100k_logit)],
})
print("1. Сравнение размера датасета:")
display(comparison_100k)

percent_100k = np.linspace(0, 1, len(test_100k_logit))
plt.figure(figsize=(10, 6))
plt.plot(percent * 100, gain_model, label="20k Logistic")
plt.plot(percent_100k * 100, gain_100k_logit, label="100k Logistic")
plt.xlabel("Процент таргетируемых клиентов (%)")
plt.ylabel("Cumulative Gain")
plt.title("Влияние размера датасета на uplift-кривую")
plt.legend()
plt.grid(True)
plt.show()

test_xgb = fit_two_model_uplift(df, xgb_factory, random_state=42)
gain_xgb = cumulative_gain_curve(test_xgb, "uplift")
xgb_comparison = pd.DataFrame({
    "model": ["LogisticRegression", "XGBoost"],
    "Uplift@10%": [
        uplift_at_k(test, "uplift", 0.10),
        uplift_at_k(test_xgb, "uplift", 0.10),
    ],
    "Uplift@30%": [
        uplift_at_k(test, "uplift", 0.30),
        uplift_at_k(test_xgb, "uplift", 0.30),
    ],
    "AUQC": [approx_auqc(gain_model), approx_auqc(gain_xgb)],
})
print("\n2. Сравнение логистической регрессии и XGBoost:")
display(xgb_comparison)

rng = np.random.default_rng(42)
test_noisy = test.copy()
noise_multiplier = np.clip(0.5 + rng.normal(0, 0.25, len(test_noisy)), 0, None)
test_noisy["noisy_uplift"] = test_noisy["uplift"] * noise_multiplier
gain_noisy = cumulative_gain_curve(test_noisy, "noisy_uplift")
auqc_noisy = approx_auqc(gain_noisy)
auqc_drop = auqc_model - auqc_noisy
print(
    f"\n3. AUQC с шумом = {auqc_noisy:.4f}; падение относительно исходной модели = {auqc_drop:.4f}"
)

k_values = [0.05, 0.10, 0.20, 0.50]
gain_for_decision = cumulative_gain_curve(test, "uplift")
k_table = pd.DataFrame({
    "k": k_values,
    "Uplift@K": [uplift_at_k(test, "uplift", k) for k in k_values],
    "Cumulative Gain@K": [gain_at_k(gain_for_decision, k) for k in k_values],
})
best_k = k_table.loc[k_table["Cumulative Gain@K"].idxmax(), "k"]
print("\n4. Uplift@K и точка остановки кампании:")
display(k_table)
print(
    f"Оптимально остановиться около {best_k:.0%}: среди проверенных долей там максимальный накопленный инкрементальный эффект."
)

test_true = test.copy()
test_true["true_uplift"] = np.where(
    (test_true["age"] < 38) & (test_true["income"] > 55000),
    0.22,
    0.035,
)
gain_true = cumulative_gain_curve(test_true, "true_uplift")

plt.figure(figsize=(10, 6))
plt.plot(percent * 100, gain_model, label="Модель")
plt.plot(percent * 100, gain_noisy, label="Модель с шумом")
plt.plot(percent * 100, gain_true, label="Истинный uplift (потолок)")
plt.plot(percent * 100, gain_random, "--", label="Случайно")
plt.xlabel("Процент таргетируемых клиентов (%)")
plt.ylabel("Cumulative Gain")
plt.title("Qini-кривые: модель, шум, случайный baseline и потолок")
plt.legend()
plt.grid(True)
plt.show()

print(f"\n5. AUQC потолка по истинному uplift = {approx_auqc(gain_true):.4f}")

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

### 4. Precision@K

**Precision@K** (точность на первых K элементах) - это доля релевантных элементов среди первых K рекомендованных/возвращенных элементов.

Precision@K = (Количество релевантных элементов в топ-K) / K

**Характеристики:**

Значение от 0 до 1

Чем выше, тем лучше

Не учитывает порядок элементов в топ-K

Полезен, когда важна точность первых рекомендаций

In [ ]:
def precision_at_k(y_true: List[int], y_pred: List[int], k: int) -> float:
    """
    Вычисляет Precision@K

    Parameters:
    -----------
    y_true : List[int]
        Список релевантных элементов (ground truth)
    y_pred : List[int]
        Список предсказанных элементов в порядке убывания релевантности
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Precision@K
    """
    # Берем первые k элементов из предсказаний
    y_pred_k = y_pred[:k]

    # Считаем, сколько из них релевантны
    relevant_count = sum(1 for item in y_pred_k if item in y_true)

    # Вычисляем precision
    precision = relevant_count / k if k > 0 else 0.0

    return precision

# Пример использования
def example_precision_at_k():
    print("Пример Precision@K:")
    print("-" * 40)

    # Пример: рекомендательная система
    y_true = [1, 3, 5]  # Релевантные элементы для пользователя
    y_pred = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # Рекомендации системы

    for k in [1, 3, 5, 10]:
        prec = precision_at_k(y_true, y_pred, k)
        print(f"Precision@{k}: {prec:.3f}")

    return y_true, y_pred

y_true_example, y_pred_example = example_precision_at_k()

In [ ]:
def visualize_precision_at_k():
    # Создаем пример данных
    y_true = [1, 3, 5, 7, 9]
    y_pred = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    # Вычисляем Precision@K для разных K
    k_values = list(range(1, 11))
    precision_values = [precision_at_k(y_true, y_pred, k) for k in k_values]

    # Создаем визуализацию
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: Precision@K для разных K
    axes[0].plot(k_values, precision_values, marker='o', linewidth=2, markersize=8)
    axes[0].set_xlabel('K', fontsize=12)
    axes[0].set_ylabel('Precision@K', fontsize=12)
    axes[0].set_title('Precision@K для разных значений K', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)

    # График 2: Иллюстрация работы Precision@K
    k_example = 5
    y_pred_k = y_pred[:k_example]
    colors = ['green' if item in y_true else 'red' for item in y_pred_k]

    axes[1].bar(range(1, k_example + 1), [1] * k_example, color=colors, edgecolor='black')
    axes[1].set_xlabel('Позиция в топ-K', fontsize=12)
    axes[1].set_ylabel('Релевантность', fontsize=12)
    axes[1].set_title(f'Precision@{k_example} = {precision_at_k(y_true, y_pred, k_example):.2f}', fontsize=14)
    axes[1].set_xticks(range(1, k_example + 1))
    axes[1].set_yticks([])
    axes[1].set_ylim(0, 1.2)

    # Добавляем подписи
    for i, (item, color) in enumerate(zip(y_pred_k, colors)):
        relevance = "Релевантный" if color == 'green' else "Не релевантный"
        axes[1].text(i + 1, 1.1, f"Элемент {item}\n({relevance})",
                    ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Выводим вычисления
    print(f"Релевантные элементы: {y_true}")
    print(f"Рекомендации: {y_pred}")
    print(f"\nPrecision@5:")
    print(f"  Релевантные в топ-5: {[item for item in y_pred[:5] if item in y_true]}")
    print(f"  Количество релевантных: {sum(1 for item in y_pred[:5] if item in y_true)}")
    print(f"  Precision@5 = {sum(1 for item in y_pred[:5] if item in y_true)} / 5 = {precision_at_k(y_true, y_pred, 5):.2f}")

visualize_precision_at_k()

**Пример из практики**

Задача рекомендации фильмов на Netflix:

Релевантные фильмы для пользователя (просмотренные и оцененные высоко): [101, 205, 308]

Система рекомендует: [101, 102, 205, 103, 104, 308, 105, 106]

Precision@3 = 2/3 ≈ 0.667 (релевантные 101 и 205 в топ-3)

Precision@5 = 2/5 = 0.4 (релевантные 101 и 205 в топ-5)

Precision@8 = 3/8 = 0.375 (все три релевантных в топ-8)


In [ ]:
def netflix_example():
    print("Пример из практики: Netflix рекомендации")
    print("=" * 50)

    # Данные
    relevant_movies = [101, 205, 308]  # Фильмы, которые пользователь любит
    recommendations = [101, 102, 205, 103, 104, 308, 105, 106]  # Рекомендации системы

    print(f"Релевантные фильмы для пользователя: {relevant_movies}")
    print(f"Рекомендации системы: {recommendations}")
    print()

    # Вычисляем Precision@K для разных K
    k_values = [3, 5, 8]
    for k in k_values:
        prec = precision_at_k(relevant_movies, recommendations, k)
        relevant_in_top_k = [movie for movie in recommendations[:k] if movie in relevant_movies]
        print(f"Precision@{k}:")
        print(f"  Релевантные в топ-{k}: {relevant_in_top_k}")
        print(f"  Precision@{k} = {len(relevant_in_top_k)}/{k} = {prec:.3f}")
        print()

netflix_example()

### 5. Recall@K

Recall@K (полнота на первых K элементах) - это доля всех релевантных элементов,
которые были найдены в первых K рекомендациях.

Recall@K = (Количество релевантных элементов в топ-K) / (Общее количество релевантных элементов)

Характеристики:

Значение от 0 до 1

Чем выше, тем лучше

Учитывает, сколько релевантных элементов мы нашли

Полезен, когда важно найти как можно больше релевантных элементов

In [ ]:
def recall_at_k(y_true: List[int], y_pred: List[int], k: int) -> float:
    """
    Вычисляет Recall@K

    Parameters:
    -----------
    y_true : List[int]
        Список релевантных элементов (ground truth)
    y_pred : List[int]
        Список предсказанных элементов в порядке убывания релевантности
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Recall@K
    """
    # Берем первые k элементов из предсказаний
    y_pred_k = y_pred[:k]

    # Считаем, сколько релевантных элементов найдено в топ-K
    relevant_found = sum(1 for item in y_pred_k if item in y_true)

    # Общее количество релевантных элементов
    total_relevant = len(y_true)

    # Вычисляем recall
    recall = relevant_found / total_relevant if total_relevant > 0 else 0.0

    return recall

# Пример использования
def example_recall_at_k():
    print("Пример Recall@K:")
    print("-" * 40)

    # Пример: поисковая система
    y_true = [1, 3, 5, 7, 9]  # Все релевантные документы
    y_pred = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # Результаты поиска

    for k in [1, 3, 5, 10]:
        rec = recall_at_k(y_true, y_pred, k)
        print(f"Recall@{k}: {rec:.3f}")

    return y_true, y_pred

y_true_recall, y_pred_recall = example_recall_at_k()

In [ ]:
def visualize_recall_at_k():
    # Создаем пример данных
    y_true = [1, 3, 5, 7, 9]
    y_pred = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    # Вычисляем Recall@K для разных K
    k_values = list(range(1, 11))
    recall_values = [recall_at_k(y_true, y_pred, k) for k in k_values]
    precision_values = [precision_at_k(y_true, y_pred, k) for k in k_values]

    # Создаем визуализацию
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: Recall@K для разных K
    axes[0].plot(k_values, recall_values, marker='o', linewidth=2, markersize=8, color='orange')
    axes[0].set_xlabel('K', fontsize=12)
    axes[0].set_ylabel('Recall@K', fontsize=12)
    axes[0].set_title('Recall@K для разных значений K', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)

    # График 2: Сравнение Precision@K и Recall@K
    axes[1].plot(k_values, precision_values, marker='o', linewidth=2, markersize=8, label='Precision@K')
    axes[1].plot(k_values, recall_values, marker='s', linewidth=2, markersize=8, label='Recall@K')
    axes[1].set_xlabel('K', fontsize=12)
    axes[1].set_ylabel('Значение метрики', fontsize=12)
    axes[1].set_title('Сравнение Precision@K и Recall@K', fontsize=14)
    axes[1].legend(fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.show()

    # Выводим вычисления для K=5
    k_example = 5
    y_pred_k = y_pred[:k_example]
    relevant_found = [item for item in y_pred_k if item in y_true]

    print(f"Все релевантные элементы: {y_true}")
    print(f"Рекомендации: {y_pred}")
    print(f"\nRecall@{k_example}:")
    print(f"  Найдено релевантных в топ-{k_example}: {relevant_found}")
    print(f"  Всего релевантных: {len(y_true)}")
    print(f"  Recall@{k_example} = {len(relevant_found)}/{len(y_true)} = {recall_at_k(y_true, y_pred, k_example):.2f}")

visualize_recall_at_k()

Пример из практики
Задача поиска документов в юридической базе:

Все релевантные документы по запросу: [doc1, doc3, doc5, doc7, doc9] (5 документов)

Поисковая система возвращает: [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8, doc9, doc10]

Recall@3 = 2/5 = 0.4 (нашли doc1 и doc3 из 5)

Recall@5 = 3/5 = 0.6 (нашли doc1, doc3, doc5 из 5)

Recall@10 = 5/5 = 1.0 (нашли все релевантные документы)

In [ ]:
def legal_search_example():
    print("Пример из практики: Поиск в юридической базе")
    print("=" * 50)

    # Данные
    relevant_docs = ['doc1', 'doc3', 'doc5', 'doc7', 'doc9']
    search_results = ['doc1', 'doc2', 'doc3', 'doc4', 'doc5', 'doc6', 'doc7', 'doc8', 'doc9', 'doc10']

    print(f"Все релевантные документы: {relevant_docs}")
    print(f"Результаты поиска: {search_results}")
    print()

    # Вычисляем Recall@K для разных K
    k_values = [3, 5, 10]
    for k in k_values:
        rec = recall_at_k(relevant_docs, search_results, k)
        relevant_found = [doc for doc in search_results[:k] if doc in relevant_docs]
        print(f"Recall@{k}:")
        print(f"  Найдено релевантных в топ-{k}: {relevant_found}")
        print(f"  Recall@{k} = {len(relevant_found)}/{len(relevant_docs)} = {rec:.3f}")
        print()

legal_search_example()

### 6. Hit Rate / Hit Ratio@K (HR@K)

Hit Rate@K (коэффициент попадания) - это бинарная метрика, которая показывает, есть ли хотя бы один релевантный элемент в топ-K рекомендаций.

Формула:

HR@K = 1, если хотя бы один релевантный элемент в топ-K

HR@K = 0, иначе

Для набора пользователей:

HR@K = (Количество пользователей с хотя бы одним попаданием) / (Общее количество пользователей)

Характеристики:

Значение от 0 до 1

Чем выше, тем лучше

Бинарная метрика (0 или 1 для одного пользователя)

Полезен для оценки, находит ли система хоть что-то релевантное

In [ ]:
def hit_rate_at_k(y_true: List[int], y_pred: List[int], k: int) -> int:
    """
    Вычисляет Hit Rate@K для одного пользователя

    Parameters:
    -----------
    y_true : List[int]
        Список релевантных элементов (ground truth)
    y_pred : List[int]
        Список предсказанных элементов в порядке убывания релевантности
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    int : 1 если есть попадание, 0 иначе
    """
    # Берем первые k элементов из предсказаний
    y_pred_k = y_pred[:k]

    # Проверяем, есть ли хотя бы один релевантный элемент
    hit = any(item in y_true for item in y_pred_k)

    return 1 if hit else 0

def hit_rate_at_k_aggregate(users_true: List[List[int]], users_pred: List[List[int]], k: int) -> float:
    """
    Вычисляет средний Hit Rate@K для набора пользователей

    Parameters:
    -----------
    users_true : List[List[int]]
        Список списков релевантных элементов для каждого пользователя
    users_pred : List[List[int]]
        Список списков предсказанных элементов для каждого пользователя
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Средний Hit Rate@K
    """
    if len(users_true) != len(users_pred):
        raise ValueError("Количество пользователей в ground truth и предсказаниях должно совпадать")

    # Вычисляем Hit Rate для каждого пользователя
    hit_rates = [hit_rate_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]

    # Вычисляем средний Hit Rate
    avg_hit_rate = sum(hit_rates) / len(hit_rates)

    return avg_hit_rate

# Пример использования
def example_hit_rate_at_k():
    print("Пример Hit Rate@K:")
    print("-" * 40)

    # Пример для одного пользователя
    y_true = [1, 3, 5]
    y_pred = [2, 4, 6, 8, 10]

    for k in [3, 5, 10]:
        hr = hit_rate_at_k(y_true, y_pred, k)
        print(f"Hit Rate@{k} для одного пользователя: {hr}")

    # Пример для нескольких пользователей
    print("\nПример для нескольких пользователей:")
    users_true = [
        [1, 3, 5],      # Пользователь 1
        [2, 4, 6],      # Пользователь 2
        [1, 2, 3],      # Пользователь 3
        [7, 8, 9]       # Пользователь 4
    ]

    users_pred = [
        [1, 2, 3, 4, 5],  # Рекомендации для пользователя 1
        [1, 3, 5, 7, 9],  # Рекомендации для пользователя 2
        [1, 4, 7, 10, 13], # Рекомендации для пользователя 3
        [10, 11, 12, 13, 14] # Рекомендации для пользователя 4
    ]

    for k in [1, 3, 5]:
        hr_avg = hit_rate_at_k_aggregate(users_true, users_pred, k)
        print(f"Средний Hit Rate@{k}: {hr_avg:.3f}")

    return users_true, users_pred

users_true_example, users_pred_example = example_hit_rate_at_k()

In [ ]:
def visualize_hit_rate_at_k():
    # Создаем пример данных для нескольких пользователей
    users_true = [
        [1, 3, 5],
        [2, 4, 6],
        [1, 2, 3],
        [7, 8, 9],
        [10, 11, 12]
    ]

    users_pred = [
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],
        [1, 4, 7, 10, 13, 16, 19, 22, 25, 28],
        [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
        [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    ]

    # Вычисляем Hit Rate@K для разных K
    k_values = list(range(1, 11))
    hr_values = [hit_rate_at_k_aggregate(users_true, users_pred, k) for k in k_values]

    # Создаем визуализацию
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: Hit Rate@K для разных K
    axes[0].plot(k_values, hr_values, marker='o', linewidth=2, markersize=8, color='green')
    axes[0].fill_between(k_values, hr_values, alpha=0.3, color='green')
    axes[0].set_xlabel('K', fontsize=12)
    axes[0].set_ylabel('Hit Rate@K', fontsize=12)
    axes[0].set_title('Hit Rate@K для разных значений K', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)

    # График 2: Hit Rate для каждого пользователя при K=5
    k_example = 5
    user_hits = [hit_rate_at_k(true, pred, k_example) for true, pred in zip(users_true, users_pred)]

    colors = ['green' if hit == 1 else 'red' for hit in user_hits]
    axes[1].bar(range(1, len(users_true) + 1), user_hits, color=colors, edgecolor='black')
    axes[1].set_xlabel('Пользователь', fontsize=12)
    axes[1].set_ylabel('Hit (1) / No Hit (0)', fontsize=12)
    axes[1].set_title(f'Hit Rate@{k_example} для каждого пользователя', fontsize=14)
    axes[1].set_xticks(range(1, len(users_true) + 1))
    axes[1].set_yticks([0, 1])
    axes[1].set_ylim(-0.1, 1.1)

    # Добавляем значения
    for i, hit in enumerate(user_hits):
        axes[1].text(i + 1, hit + 0.05, f"{hit}", ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Выводим детальную информацию
    print(f"Средний Hit Rate@{k_example}: {np.mean(user_hits):.3f}")
    print(f"Пользователи с попаданием: {[i+1 for i, hit in enumerate(user_hits) if hit == 1]}")
    print(f"Пользователи без попадания: {[i+1 for i, hit in enumerate(user_hits) if hit == 0]}")

visualize_hit_rate_at_k()

**Пример из практики**

Задача рекомендации товаров в интернет-магазине:

Для 5 пользователей система делает рекомендации

Hit Rate@5 показывает, скольким пользователям понравился хотя бы один рекомендованный товар

Если 3 из 5 пользователей имеют хотя бы один релевантный товар в топ-5, то HR@5 = 0.6


In [ ]:
def ecommerce_example():
    print("Пример из практики: Рекомендации в интернет-магазине")
    print("=" * 50)

    # Данные: покупки пользователей (релевантные товары)
    users_purchases = [
        [101, 102, 103],  # Пользователь 1 купил товары 101, 102, 103
        [201, 202],       # Пользователь 2 купил товары 201, 202
        [301, 302, 303],  # Пользователь 3 купил товары 301, 302, 303
        [401],            # Пользователь 4 купил товар 401
        [501, 502]        # Пользователь 5 купил товары 501, 502
    ]

    # Рекомендации системы
    users_recommendations = [
        [101, 110, 120, 130, 140],  # Для пользователя 1
        [210, 220, 230, 240, 250],  # Для пользователя 2
        [301, 310, 320, 330, 340],  # Для пользователя 3
        [410, 420, 430, 440, 450],  # Для пользователя 4
        [501, 510, 520, 530, 540]   # Для пользователя 5
    ]

    print("Релевантные товары (покупки) для каждого пользователя:")
    for i, purchases in enumerate(users_purchases, 1):
        print(f"  Пользователь {i}: {purchases}")

    print("\nРекомендации системы (топ-5) для каждого пользователя:")
    for i, recs in enumerate(users_recommendations, 1):
        print(f"  Пользователь {i}: {recs}")

    print("\nHit Rate@5:")
    for i, (purchases, recs) in enumerate(zip(users_purchases, users_recommendations), 1):
        hit = hit_rate_at_k(purchases, recs, 5)
        hit_items = [item for item in recs[:5] if item in purchases]
        status = "ЕСТЬ попадание" if hit == 1 else "НЕТ попадания"
        print(f"  Пользователь {i}: {status} {f'(попался товар {hit_items[0]})' if hit_items else ''}")

    hr5 = hit_rate_at_k_aggregate(users_purchases, users_recommendations, 5)
    print(f"\nСредний Hit Rate@5: {hr5:.3f} ({int(hr5 * len(users_purchases))}/{len(users_purchases)} пользователей)")

ecommerce_example()

### 7. Mean Average Precision (MAP@K)

**Mean Average Precision (MAP@K)** - это среднее значение Average Precision (AP) по всем пользователям/запросам.

Average Precision (AP) для одного пользователя/запроса:

Для каждой позиции k, где найден релевантный элемент, вычисляется Precision@k
AP - это среднее значение этих Precision@k


**Формула для одного пользователя:**

AP = (1 / количество релевантных элементов) * Σ_{k=1}^{K} Precision@k * rel_k

где rel_k = 1, если элемент на позиции k релевантен, иначе 0

**MAP@K - это среднее AP по всем пользователям:**

MAP@K = (1 / количество пользователей) * Σ_{i=1}^{N} AP_i

Характеристики:

Значение от 0 до 1

Чем выше, тем лучше

Учитывает и точность, и порядок релевантных элементов

Широко используется в информационном поиске и рекомендательных системах

In [ ]:
def average_precision_at_k(y_true: List[int], y_pred: List[int], k: int) -> float:
    """
    Вычисляет Average Precision@K для одного пользователя

    Parameters:
    -----------
    y_true : List[int]
        Список релевантных элементов (ground truth)
    y_pred : List[int]
        Список предсказанных элементов в порядке убывания релевантности
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Average Precision@K
    """
    # Берем первые k элементов
    y_pred_k = y_pred[:k]

    # Инициализируем переменные
    relevant_count = 0
    sum_precision = 0.0

    # Вычисляем precision на каждой позиции, где найден релевантный элемент
    for i, item in enumerate(y_pred_k, 1):
        if item in y_true:
            relevant_count += 1
            precision_at_i = relevant_count / i
            sum_precision += precision_at_i

    # Вычисляем Average Precision
    ap = sum_precision / min(len(y_true), k) if min(len(y_true), k) > 0 else 0.0

    return ap

def mean_average_precision_at_k(users_true: List[List[int]], users_pred: List[List[int]], k: int) -> float:
    """
    Вычисляет Mean Average Precision@K для набора пользователей

    Parameters:
    -----------
    users_true : List[List[int]]
        Список списков релевантных элементов для каждого пользователя
    users_pred : List[List[int]]
        Список списков предсказанных элементов для каждого пользователя
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Mean Average Precision@K
    """
    if len(users_true) != len(users_pred):
        raise ValueError("Количество пользователей в ground truth и предсказаниях должно совпадать")

    # Вычисляем AP для каждого пользователя
    aps = [average_precision_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]

    # Вычисляем среднее значение
    map_score = sum(aps) / len(aps) if len(aps) > 0 else 0.0

    return map_score

# Пример использования
def example_map_at_k():
    print("Пример MAP@K:")
    print("-" * 40)

    # Пример для одного пользователя
    y_true = [1, 3, 5]
    y_pred = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    ap = average_precision_at_k(y_true, y_pred, 5)
    print(f"Average Precision@5 для одного пользователя: {ap:.3f}")

    # Подробный расчет
    print("\nПодробный расчет AP@5:")
    print("Позиция | Элемент | Релевантен? | Количество релевантных | Precision@k")
    print("-" * 70)

    relevant_count = 0
    for i, item in enumerate(y_pred[:5], 1):
        is_relevant = item in y_true
        if is_relevant:
            relevant_count += 1
            precision = relevant_count / i
            print(f"{i:7d} | {item:7d} | {'Да':11s} | {relevant_count:21d} | {precision:.3f}")
        else:
            print(f"{i:7d} | {item:7d} | {'Нет':11s} | {relevant_count:21d} | -")

    print(f"\nAP@5 = (1/{min(len(y_true), 5)}) * (Сумма Precision@k для релевантных позиций)")
    print(f"     = (1/3) * ({1/1} + {2/3})")
    print(f"     = (1/3) * (1.000 + 0.667)")
    print(f"     = (1/3) * 1.667 = {ap:.3f}")

    # Пример для нескольких пользователей
    print("\nПример для нескольких пользователей:")
    users_true = [
        [1, 3, 5],
        [2, 4],
        [1, 2, 3]
    ]

    users_pred = [
        [1, 2, 3, 4, 5],
        [1, 3, 5, 7, 9],
        [1, 4, 7, 10, 13]
    ]

    map_score = mean_average_precision_at_k(users_true, users_pred, 5)
    print(f"MAP@5: {map_score:.3f}")

    return users_true, users_pred

users_true_map, users_pred_map = example_map_at_k()

In [ ]:
def visualize_map_at_k():
    # Создаем пример данных
    users_true = [
        [1, 3, 5, 7],
        [2, 4, 6],
        [1, 2, 3, 4, 5]
    ]

    users_pred = [
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],
        [1, 4, 7, 10, 13, 16, 19, 22, 25, 28]
    ]

    # Вычисляем MAP@K для разных K
    k_values = list(range(1, 11))
    map_values = [mean_average_precision_at_k(users_true, users_pred, k) for k in k_values]

    # Вычисляем AP@K для каждого пользователя при K=5
    k_example = 5
    ap_values = [average_precision_at_k(true, pred, k_example) for true, pred in zip(users_true, users_pred)]

    # Создаем визуализацию
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: MAP@K для разных K
    axes[0].plot(k_values, map_values, marker='o', linewidth=2, markersize=8, color='purple')
    axes[0].fill_between(k_values, map_values, alpha=0.3, color='purple')
    axes[0].set_xlabel('K', fontsize=12)
    axes[0].set_ylabel('MAP@K', fontsize=12)
    axes[0].set_title('Mean Average Precision (MAP) для разных K', fontsize=14)
    axes[0].grid(True, alpha=0)
    axes[0].set_ylim(-0.05, 1.05)

    # График 2: AP@K для каждого пользователя при K=5
    users_labels = [f'Пользователь {i+1}' for i in range(len(users_true))]
    colors = plt.cm.Set3(np.linspace(0, 1, len(users_true)))

    bars = axes[1].bar(users_labels, ap_values, color=colors, edgecolor='black')
    axes[1].set_xlabel('Пользователь', fontsize=12)
    axes[1].set_ylabel('Average Precision@5', fontsize=12)
    axes[1].set_title(f'AP@5 для каждого пользователя\nMAP@5 = {np.mean(ap_values):.3f}', fontsize=14)
    axes[1].set_ylim(0, 1.1)

    # Добавляем значения на столбцы
    for bar, ap in zip(bars, ap_values):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{ap:.3f}', ha='center', va='bottom', fontsize=11)

    plt.tight_layout()
    plt.show()

    # Выводим детальные расчеты для первого пользователя
    print("Детальный расчет для Пользователя 1:")
    print(f"Релевантные элементы: {users_true[0]}")
    print(f"Рекомендации: {users_pred[0][:k_example]}")
    print("\nРасчет AP@5:")

    relevant_count = 0
    sum_precision = 0
    for i, item in enumerate(users_pred[0][:k_example], 1):
        if item in users_true[0]:
            relevant_count += 1
            precision_at_i = relevant_count / i
            sum_precision += precision_at_i
            print(f"  Позиция {i}: элемент {item} - релевантный, Precision@{i} = {relevant_count}/{i} = {precision_at_i:.3f}")
        else:
            print(f"  Позиция {i}: элемент {item} - не релевантный")

    ap = sum_precision / min(len(users_true[0]), k_example)
    print(f"\n  AP@5 = (1/{min(len(users_true[0]), k_example)}) * Σ Precision@k для релевантных позиций")
    print(f"       = (1/{min(len(users_true[0]), k_example)}) * {sum_precision:.3f}")
    print(f"       = {ap:.3f}")

visualize_map_at_k()

**Пример из практики**

Задача ранжирования поисковых результатов:

У нас есть 3 поисковых запроса (пользователя)

Для каждого запроса есть релевантные документы

Система возвращает ранжированный список документов

MAP@10 оценивает, насколько хорошо система находит и правильно ранжирует релевантные документы

In [ ]:
def search_engine_example():
    print("Пример из практики: Поисковая система")
    print("=" * 50)

    # Данные: релевантные документы для 3 поисковых запросов
    queries_relevant = [
        ['doc1', 'doc3', 'doc5', 'doc7'],  # Запрос 1
        ['doc2', 'doc4', 'doc6'],          # Запрос 2
        ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']  # Запрос 3
    ]

    # Результаты поиска для каждого запроса
    search_results = [
        ['doc1', 'doc2', 'doc3', 'doc4', 'doc5', 'doc6', 'doc7', 'doc8', 'doc9', 'doc10'],
        ['doc1', 'doc3', 'doc5', 'doc7', 'doc9', 'doc11', 'doc13', 'doc15', 'doc17', 'doc19'],
        ['doc1', 'doc4', 'doc7', 'doc10', 'doc13', 'doc16', 'doc19', 'doc22', 'doc25', 'doc28']
    ]

    print("Релевантные документы для каждого запроса:")
    for i, relevant in enumerate(queries_relevant, 1):
        print(f"  Запрос {i}: {relevant}")

    print("\nРезультаты поиска (первые 5):")
    for i, results in enumerate(search_results, 1):
        print(f"  Запрос {i}: {results[:5]}")

    # Вычисляем AP@5 для каждого запроса
    print("\nРасчет Average Precision@5 для каждого запроса:")
    aps = []
    for i, (relevant, results) in enumerate(zip(queries_relevant, search_results), 1):
        ap = average_precision_at_k(relevant, results, 5)
        aps.append(ap)

        # Находим релевантные документы в топ-5
        relevant_in_top5 = [doc for doc in results[:5] if doc in relevant]

        print(f"\nЗапрос {i}:")
        print(f"  Релевантные в топ-5: {relevant_in_top5}")
        print(f"  AP@5 = {ap:.3f}")

    # Вычисляем MAP@5
    map5 = mean_average_precision_at_k(queries_relevant, search_results, 5)
    print(f"\nMAP@5 = Среднее AP@5 по всем запросам")
    print(f"      = ({' + '.join([f'{ap:.3f}' for ap in aps])}) / {len(aps)}")
    print(f"      = {sum(aps):.3f} / {len(aps)} = {map5:.3f}")

    # Сравниваем с другими метриками
    print("\nСравнение с другими метриками для K=5:")
    for i, (relevant, results) in enumerate(zip(queries_relevant, search_results), 1):
        prec = precision_at_k(relevant, results, 5)
        rec = recall_at_k(relevant, results, 5)
        ap = average_precision_at_k(relevant, results, 5)
        print(f"Запрос {i}: Precision@5={prec:.3f}, Recall@5={rec:.3f}, AP@5={ap:.3f}")

search_engine_example()

### 8.Mean Reciprocal Rank (MRR / MRR@K)

**Mean Reciprocal Rank (MRR)** - это среднее обратных рангов первых релевантных элементов по всем пользователям/запросам.

Для одного пользователя/запроса:

RR = 1 / rank первого релевантного элемента

где rank - позиция первого релевантного элемента (начиная с 1)

Если релевантных элементов нет, RR = 0

Для набора пользователей:

MRR = (1 / N) * Σ_{i=1}^{N} RR_i

MRR@K - то же самое, но рассматриваются только первые K позиций.

Характеристики:

Значение от 0 до 1

Чем выше, тем лучше

Учитывает позицию первого релевантного элемента

Полезен, когда важно быстро найти первый релевантный результат

In [ ]:
def reciprocal_rank_at_k(y_true: List[int], y_pred: List[int], k: int) -> float:
    """
    Вычисляет Reciprocal Rank@K для одного пользователя

    Parameters:
    -----------
    y_true : List[int]
        Список релевантных элементов (ground truth)
    y_pred : List[int]
        Список предсказанных элементов в порядке убывания релевантности
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Reciprocal Rank@K
    """
    # Берем первые k элементов
    y_pred_k = y_pred[:k]

    # Ищем первый релевантный элемент
    for i, item in enumerate(y_pred_k, 1):
        if item in y_true:
            return 1.0 / i

    # Если релевантных элементов нет в топ-K
    return 0.0

def mean_reciprocal_rank_at_k(users_true: List[List[int]], users_pred: List[List[int]], k: int) -> float:
    """
    Вычисляет Mean Reciprocal Rank@K для набора пользователей

    Parameters:
    -----------
    users_true : List[List[int]]
        Список списков релевантных элементов для каждого пользователя
    users_pred : List[List[int]]
        Список списков предсказанных элементов для каждого пользователя
    k : int
        Количество элементов для рассмотрения

    Returns:
    --------
    float : Mean Reciprocal Rank@K
    """
    if len(users_true) != len(users_pred):
        raise ValueError("Количество пользователей в ground truth и предсказаниях должно совпадать")

    # Вычисляем RR для каждого пользователя
    rrs = [reciprocal_rank_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]

    # Вычисляем среднее значение
    mrr = sum(rrs) / len(rrs) if len(rrs) > 0 else 0.0

    return mrr

# Пример использования
def example_mrr_at_k():
    print("Пример MRR@K:")
    print("-" * 40)

    # Пример для одного пользователя
    y_true = [1, 3, 5]
    y_pred = [2, 4, 1, 6, 8, 3, 5, 7, 9, 10]

    for k in [3, 5, 10]:
        rr = reciprocal_rank_at_k(y_true, y_pred, k)
        print(f"Reciprocal Rank@{k} для одного пользователя: {rr:.3f}")

    # Подробный расчет для K=5
    print("\nПодробный расчет RR@5:")
    print("Позиция | Элемент | Релевантен?")
    print("-" * 35)

    for i, item in enumerate(y_pred[:5], 1):
        is_relevant = item in y_true
        print(f"{i:7d} | {item:7d} | {'Да' if is_relevant else 'Нет'}")

    # Находим первый релевантный элемент
    for i, item in enumerate(y_pred[:5], 1):
        if item in y_true:
            print(f"\nПервый релевантный элемент найден на позиции {i}")
            print(f"RR@5 = 1/{i} = {1.0/i:.3f}")
            break

    # Пример для нескольких пользователей
    print("\nПример для нескольких пользователей:")
    users_true = [
        [1, 3, 5],
        [2, 4],
        [1, 2, 3],
        [7, 8, 9]
    ]

    users_pred = [
        [1, 2, 3, 4, 5],
        [1, 3, 5, 7, 9],
        [4, 7, 10, 13, 1],
        [10, 11, 12, 13, 14]
    ]

    mrr = mean_reciprocal_rank_at_k(users_true, users_pred, 5)
    print(f"MRR@5: {mrr:.3f}")

    return users_true, users_pred

users_true_mrr, users_pred_mrr = example_mrr_at_k()

In [ ]:
def visualize_mrr_at_k():
    # Создаем пример данных
    users_true = [
        [1, 3, 5, 7],
        [2, 4, 6],
        [1, 2, 3, 4, 5],
        [10, 20, 30],
        [5, 10, 15]
    ]

    users_pred = [
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],
        [4, 7, 10, 13, 16, 19, 22, 25, 28, 31],
        [5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
        [1, 2, 3, 4, 6, 7, 8, 9, 11, 12]
    ]

    # Вычисляем MRR@K для разных K
    k_values = list(range(1, 11))
    mrr_values = [mean_reciprocal_rank_at_k(users_true, users_pred, k) for k in k_values]

    # Вычисляем RR@K для каждого пользователя при K=5
    k_example = 5
    rr_values = [reciprocal_rank_at_k(true, pred, k_example) for true, pred in zip(users_true, users_pred)]

    # Находим позиции первых релевантных элементов
    first_relevant_positions = []
    for true, pred in zip(users_true, users_pred):
        position = None
        for i, item in enumerate(pred[:k_example], 1):
            if item in true:
                position = i
                break
        first_relevant_positions.append(position if position is not None else 0)

    # Создаем визуализацию
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: MRR@K для разных K
    axes[0].plot(k_values, mrr_values, marker='o', linewidth=2, markersize=8, color='brown')
    axes[0].fill_between(k_values, mrr_values, alpha=0.3, color='brown')
    axes[0].set_xlabel('K', fontsize=12)
    axes[0].set_ylabel('MRR@K', fontsize=12)
    axes[0].set_title('Mean Reciprocal Rank (MRR) для разных K', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)

    # График 2: Позиции первых релевантных элементов и RR@5
    x = np.arange(len(users_true))
    width = 0.35

    # Бар для позиции первого релевантного элемента
    bars1 = axes[1].bar(x - width/2, first_relevant_positions, width,
                       label='Позиция первого релевантного', color='lightblue', edgecolor='black')

    # Бар для Reciprocal Rank
    bars2 = axes[1].bar(x + width/2, rr_values, width,
                       label='Reciprocal Rank@5', color='lightcoral', edgecolor='black')

    axes[1].set_xlabel('Пользователь', fontsize=12)
    axes[1].set_ylabel('Значение', fontsize=12)
    axes[1].set_title('Позиция первого релевантного элемента и RR@5', fontsize=14)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'Пользователь {i+1}' for i in range(len(users_true))])
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3, axis='y')

    # Добавляем значения на столбцы
    for bar, pos in zip(bars1, first_relevant_positions):
        height = bar.get_height()
        if height > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                        f'Поз. {pos}', ha='center', va='bottom', fontsize=10)

    for bar, rr in zip(bars2, rr_values):
        height = bar.get_height()
        if height > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                        f'{rr:.2f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Выводим детальную информацию
    print(f"MRR@{k_example}: {np.mean(rr_values):.3f}")
    print("\nДетали по пользователям:")
    for i, (true, pred, pos, rr) in enumerate(zip(users_true, users_pred, first_relevant_positions, rr_values), 1):
        if pos > 0:
            first_relevant = [item for item in pred[:pos] if item in true][0]
            print(f"Пользователь {i}: первый релевантный '{first_relevant}' на позиции {pos}, RR@5 = 1/{pos} = {rr:.3f}")
        else:
            print(f"Пользователь {i}: нет релевантных в топ-5, RR@5 = 0")

visualize_mrr_at_k()

**Пример из практики**
Задача голосового помощника (например, Алиса, Siri):

Пользователь задает вопрос

Система возвращает несколько возможных ответов

MRR оценивает, насколько быстро система находит правильный ответ

In [ ]:
def voice_assistant_example():
    print("Пример из практики: Голосовой помощник")
    print("=" * 50)

    # Данные: вопросы пользователей и правильные ответы
    user_questions = [
        "Какая погода в Москве?",           # Вопрос 1
        "Кто президент США?",               # Вопрос 2
        "Сколько будет 2+2?",               # Вопрос 3
        "Какой сегодня день?",              # Вопрос 4
        "Включи музыку"                     # Вопрос 5
    ]

    # Правильные ответы (могут быть несколько вариантов)
    correct_answers = [
        ["Погода в Москве: +20°C, солнечно", "В Москве +20 градусов"],
        ["Джо Байден", "Президент США - Джо Байден"],
        ["4", "Два плюс два равно четыре"],
        ["Сегодня понедельник, 15 мая", "15 мая, понедельник"],
        ["Включаю музыку", "Запускаю музыкальный плеер"]
    ]

    # Ответы системы (ранжированные)
    system_responses = [
        ["Погода в Москве: +20°C, солнечно", "В Москве хорошая погода", "Сейчас в Москве", "Московская погода"],
        ["Барак Обама", "Джо Байден", "Дональд Трамп", "Президент США"],
        ["5", "4", "22", "2+2=4"],
        ["Сегодня вторник", "15 мая", "Понедельник, 15 мая", "Сегодня 15 мая"],
        ["Не понимаю команду", "Что включить?", "Включаю видео", "Запускаю музыку"]
    ]

    print("Вопросы и ответы системы (первые 3):")
    for i, (question, responses, answers) in enumerate(zip(user_questions, system_responses, correct_answers), 1):
        print(f"\nВопрос {i}: '{question}'")
        print(f"  Ответы системы: {responses[:3]}")
        print(f"  Правильные ответы: {answers[:2]}")

    # Преобразуем в числовые ID для вычисления метрик
    # Для простоты будем считать, что каждый ответ имеет уникальный ID
    users_true_ids = []
    users_pred_ids = []

    for answers, responses in zip(correct_answers, system_responses):
        # ID правильных ответов (используем весь ответ как идентификатор)
        true_ids = [hash(answer) % 100 for answer in answers]

        # ID ответов системы
        pred_ids = [hash(response) % 100 for response in responses]

        users_true_ids.append(true_ids)
        users_pred_ids.append(pred_ids)

    # Вычисляем MRR@3 (первые 3 ответа)
    mrr3 = mean_reciprocal_rank_at_k(users_true_ids, users_pred_ids, 3)

    print("\n\nОценка MRR@3 (первые 3 ответа):")
    for i, (question, true_ids, pred_ids) in enumerate(zip(user_questions, users_true_ids, users_pred_ids), 1):
        rr = reciprocal_rank_at_k(true_ids, pred_ids, 3)

        # Находим позицию первого правильного ответа
        position = None
        for j, pred_id in enumerate(pred_ids[:3], 1):
            if pred_id in true_ids:
                position = j
                break

        if position:
            print(f"Вопрос {i}: первый правильный ответ на позиции {position}, RR@3 = 1/{position} = {rr:.3f}")
        else:
            print(f"Вопрос {i}: нет правильных ответов в топ-3, RR@3 = 0")

    print(f"\nMRR@3 = Среднее RR@3 по всем вопросам = {mrr3:.3f}")

    # Сравниваем с Hit Rate@3
    hr3 = hit_rate_at_k_aggregate(users_true_ids, users_pred_ids, 3)
    print(f"Hit Rate@3 = {hr3:.3f} ({int(hr3 * len(user_questions))}/{len(user_questions)} вопросов имеют правильный ответ в топ-3)")

voice_assistant_example()

In [ ]:
def compare_all_metrics():
    print("Сравнение всех метрик на одном примере")
    print("=" * 60)

    # Создаем пример данных
    users_true = [
        [1, 3, 5, 7],      # Пользователь 1
        [2, 4, 6],         # Пользователь 2
        [1, 2, 3, 4, 5]    # Пользователь 3
    ]

    users_pred = [
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],  # Рекомендации для пользователя 1
        [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],  # Для пользователя 2
        [1, 4, 7, 10, 13, 16, 19, 22, 25, 28]  # Для пользователя 3
    ]

    k = 5

    print(f"Оценка для K = {k}")
    print("-" * 60)

    # Вычисляем все метрики
    metrics = {}

    # Precision@K
    precisions = [precision_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]
    metrics['Precision@K'] = np.mean(precisions)

    # Recall@K
    recalls = [recall_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]
    metrics['Recall@K'] = np.mean(recalls)

    # Hit Rate@K
    hit_rates = [hit_rate_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]
    metrics['Hit Rate@K'] = np.mean(hit_rates)

    # MAP@K
    aps = [average_precision_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]
    metrics['MAP@K'] = np.mean(aps)

    # MRR@K
    rrs = [reciprocal_rank_at_k(true, pred, k) for true, pred in zip(users_true, users_pred)]
    metrics['MRR@K'] = np.mean(rrs)

    # Создаем таблицу сравнения
    df_comparison = pd.DataFrame({
        'Метрика': list(metrics.keys()),
        'Значение': list(metrics.values()),
        'Описание': [
            'Доля релевантных среди первых K',
            'Доля найденных релевантных от общего числа',
            'Доля пользователей с хотя бы одним попаданием',
            'Средняя точность с учетом порядка',
            'Среднее обратных рангов первых релевантных'
        ]
    })

    print(df_comparison.to_string(index=False))

    # Визуализация сравнения
    fig, ax = plt.subplots(figsize=(12, 6))

    metrics_names = list(metrics.keys())
    metrics_values = list(metrics.values())

    colors = plt.cm.Set3(np.linspace(0, 1, len(metrics)))
    bars = ax.bar(metrics_names, metrics_values, color=colors, edgecolor='black')

    ax.set_xlabel('Метрика', fontsize=12)
    ax.set_ylabel('Значение', fontsize=12)
    ax.set_title(f'Сравнение метрик ранжирования для K={k}', fontsize=14)
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y')

    # Добавляем значения на столбцы
    for bar, value in zip(bars, metrics_values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{value:.3f}', ha='center', va='bottom', fontsize=11)

    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Выводы
    print("\nКлючевые выводы:")
    print("1. Precision@K и Recall@K - базовые метрики, но не учитывают порядок")
    print("2. Hit Rate@K - простая бинарная метрика, хороша для быстрой оценки")
    print("3. MAP@K - учитывает порядок релевантных элементов")
    print("4. MRR@K - фокусируется на позиции первого релевантного элемента")

    print("\nРекомендации по выбору метрики:")
    print("- Для простых задач: Precision@K, Recall@K")
    print("- Для рекомендательных систем: Hit Rate@K, MAP@K")
    print("- Для поисковых систем: MAP@K")
    print("- Для вопросно-ответных систем: MRR@K")

compare_all_metrics()

## Сравнение метрик

### Задания на закрепление материала

Задание 1: Реализация метрик с нуля

In [ ]:
def task_1_implementation():
    """
    Задание 1: Реализуйте все метрики с нуля без использования готовых функций
    """
    print("Задание 1: Реализация метрик с нуля")
    print("=" * 50)

    # Данные для тестирования
    y_true = [1, 3, 5, 7, 9]
    y_pred = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    relevances = [3, 2, 3, 0, 1, 2, 3, 2, 0, 1]

    # 1. Реализуйте Precision@K
    def my_precision_at_k(true, pred, k):
        if k <= 0:
            return 0.0
        true_set = set(true)
        pred_k = pred[:k]
        return sum(item in true_set for item in pred_k) / k

    # 2. Реализуйте Recall@K
    def my_recall_at_k(true, pred, k):
        if not true:
            return 0.0
        true_set = set(true)
        pred_k = pred[:k]
        return sum(item in true_set for item in pred_k) / len(true_set)

    # 3. Реализуйте Average Precision@K
    def my_average_precision_at_k(true, pred, k):
        true_set = set(true)
        if not true_set or k <= 0:
            return 0.0

        score = 0.0
        hits = 0
        for i, item in enumerate(pred[:k], start=1):
            if item in true_set:
                hits += 1
                score += hits / i

        return score / min(len(true_set), k)

    # Тестирование
    k = 5
    print(f"Тестирование для K={k}:")
    print(f"Precision@{k}: {my_precision_at_k(y_true, y_pred, k):.3f}")
    print(f"Recall@{k}: {my_recall_at_k(y_true, y_pred, k):.3f}")
    print(f"AP@{k}: {my_average_precision_at_k(y_true, y_pred, k):.3f}")

    # Сравнение с реализованными ранее функциями
    print("\nСравнение с готовыми функциями:")
    print(f"Precision@{k}: готовый={precision_at_k(y_true, y_pred, k):.3f}, ваш={my_precision_at_k(y_true, y_pred, k):.3f}")
    print(f"Recall@{k}: готовый={recall_at_k(y_true, y_pred, k):.3f}, ваш={my_recall_at_k(y_true, y_pred, k):.3f}")
    print(f"AP@{k}: готовый={average_precision_at_k(y_true, y_pred, k):.3f}, ваш={my_average_precision_at_k(y_true, y_pred, k):.3f}")

task_1_implementation()

### Задание 2: Анализ рекомендательной системы

In [ ]:
def task_2_analysis():
    """
    Задание 2: Проанализируйте рекомендательную систему
    """
    print("\n" + "="*60)
    print("Задание 2: Анализ рекомендательной системы")
    print("=" * 60)

    # Данные рекомендательной системы
    # 5 пользователей, система рекомендует 10 товаров каждому
    users_true = [
        [101, 103, 105],  # Пользователь 1 купил товары 101, 103, 105
        [202, 204],       # Пользователь 2 купил товары 202, 204
        [301, 303, 305, 307],  # Пользователь 3
        [401, 403],       # Пользователь 4
        [501, 503, 505, 507, 509]  # Пользователь 5
    ]

    users_pred = [
        [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],  # Для пользователя 1
        [201, 202, 203, 204, 205, 206, 207, 208, 209, 210],  # Для пользователя 2
        [301, 302, 303, 304, 305, 306, 307, 308, 309, 310],  # Для пользователя 3
        [401, 402, 403, 404, 405, 406, 407, 408, 409, 410],  # Для пользователя 4
        [501, 502, 503, 504, 505, 506, 507, 508, 509, 510]   # Для пользователя 5
    ]

    # Данные для NDCG (оценки товаров пользователями 1-5)
    users_relevances = [
        [3, 2, 3, 1, 2, 0, 1, 0, 1, 0],  # Пользователь 1
        [2, 3, 1, 2, 0, 1, 0, 1, 0, 0],  # Пользователь 2
        [3, 1, 3, 2, 3, 1, 2, 0, 1, 0],  # Пользователь 3
        [2, 1, 3, 0, 1, 0, 1, 0, 0, 0],  # Пользователь 4
        [3, 2, 3, 1, 3, 2, 3, 1, 2, 0]   # Пользователь 5
    ]

    print("Данные рекомендательной системы:")
    print(f"Количество пользователей: {len(users_true)}")
    print(f"Количество рекомендаций на пользователя: {len(users_pred[0])}")

    # Задачи:
    print("\nЗадачи для анализа:")
    print("1. Вычислите Precision@3, Precision@5, Precision@10 для системы")
    print("2. Вычислите Recall@3, Recall@5, Recall@10")
    print("3. Вычислите Hit Rate@5")
    print("4. Вычислите MAP@5 и MAP@10")
    print("5. Вычислите MRR@5")
    print("6. Вычислите NDCG@5")
    print("7. Проанализируйте, какие пользователи получают лучшие рекомендации")
    print("8. Предложите, как улучшить систему")

    def dcg_at_k(relevances, k):
        rel = np.asarray(relevances[:k], dtype=float)
        if len(rel) == 0:
            return 0.0
        discounts = np.log2(np.arange(2, len(rel) + 2))
        return float(np.sum((2 ** rel - 1) / discounts))

    def ndcg_at_k(relevances, k):
        actual = dcg_at_k(relevances, k)
        ideal = dcg_at_k(sorted(relevances, reverse=True), k)
        return actual / ideal if ideal > 0 else 0.0

    aggregate_rows = []
    for k in [3, 5, 10]:
        aggregate_rows.append({
            "K": k,
            "Precision@K": np.mean([precision_at_k(t, p, k) for t, p in zip(users_true, users_pred)]),
            "Recall@K": np.mean([recall_at_k(t, p, k) for t, p in zip(users_true, users_pred)]),
        })

    aggregate_table = pd.DataFrame(aggregate_rows)
    print("\n1-2. Средние Precision@K и Recall@K:")
    display(aggregate_table)

    hit_rate_5 = np.mean([hit_rate_at_k(t, p, 5) for t, p in zip(users_true, users_pred)])
    map_5 = mean_average_precision_at_k(users_true, users_pred, 5)
    map_10 = mean_average_precision_at_k(users_true, users_pred, 10)
    mrr_5 = mean_reciprocal_rank_at_k(users_true, users_pred, 5)
    mean_ndcg_5 = np.mean([ndcg_at_k(rel, 5) for rel in users_relevances])

    summary_table = pd.DataFrame({
        "Метрика": ["Hit Rate@5", "MAP@5", "MAP@10", "MRR@5", "NDCG@5"],
        "Значение": [hit_rate_5, map_5, map_10, mrr_5, mean_ndcg_5],
    })
    print("\n3-6. Итоговые метрики системы:")
    display(summary_table)

    user_rows = []
    for idx, (true_items, pred_items, relevances) in enumerate(zip(users_true, users_pred, users_relevances), start=1):
        user_rows.append({
            "Пользователь": idx,
            "Precision@5": precision_at_k(true_items, pred_items, 5),
            "Recall@5": recall_at_k(true_items, pred_items, 5),
            "AP@5": average_precision_at_k(true_items, pred_items, 5),
            "RR@5": reciprocal_rank_at_k(true_items, pred_items, 5),
            "NDCG@5": ndcg_at_k(relevances, 5),
        })

    user_table = pd.DataFrame(user_rows)
    user_table["Средний скор"] = user_table[["Precision@5", "Recall@5", "AP@5", "RR@5", "NDCG@5"]].mean(axis=1)
    print("\n7. Метрики по пользователям:")
    display(user_table)

    best_user = int(user_table.loc[user_table["Средний скор"].idxmax(), "Пользователь"])
    weakest_user = int(user_table.loc[user_table["Средний скор"].idxmin(), "Пользователь"])
    print(
        f"Лучшие рекомендации получает пользователь {best_user}: у него высокий ранний hit, AP@5 и NDCG@5. "
        f"Слабее всего качество у пользователя {weakest_user}, поэтому для похожих профилей стоит улучшать ранжирование первых позиций."
    )

    print("\n8. Как улучшить систему:")
    print("- оптимизировать модель под ранние позиции: MAP@K/NDCG@K, а не только общую точность;")
    print("- добавить признаки пользователя и товара, чтобы лучше различать релевантные элементы в топ-5;")
    print("- персонализировать длину списка рекомендаций: пользователям с малым recall показывать больше кандидатов;")
    print("- регулярно пересчитывать ranking на свежих покупках и оценках, чтобы релевантные товары поднимались выше.")


task_2_analysis()